# 02 — TRIBE inference over the stimulus corpus

Track A, Colab GPU.

> ## ⚠️ Commit results before the session closes
>
> Colab sessions die at 12h, on disconnect, and on idle. Predictions are
> written to **Drive** inside the loop, so they survive — but the manifest
> summary and any repo-side artifact do not. Before you close this notebook:
>
> 1. run the last cell (manifest verification),
> 2. copy `manifest.parquet` off the runtime if you changed cache roots,
> 3. commit and push any repo changes from a machine with push access.
>
> The run is idempotent: a stimulus already in the cache is skipped, never
> regenerated (§4.3). Re-running after a dropped session resumes; it does not
> restart.

Nothing below computes anything itself.


## Bootstrap (run once per session, then **Runtime → Restart session**)

Order matters and is not cosmetic:

1. **GPU check** — Track A needs one. Runtime → Change runtime type → GPU.
2. **Mount Drive** — Colab has no persistent disk. Sessions die at 12h, on
   disconnect, or on idle.
3. **Set `HF_HOME` before importing anything from HuggingFace.** Once a HF
   module is imported the cache location is fixed for the process, and the
   ~1 GB checkpoint lands on the ephemeral runtime disk instead of Drive.
4. **Clone the repo and `uv pip install --system`** — into Colab's own
   interpreter, never `uv sync` into a separate venv the notebook cannot see.
5. **Authenticate** from Colab Secrets (🔑), never from a literal in a cell.

Everything after that is a single call into a script in `scripts/`. No project
logic lives in this notebook: a cell dies with the session, and brief §4.3
requires every result to come from a script in the repo.


In [ ]:
# 1. GPU check
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "NO GPU")


In [ ]:
# 2. Mount Drive (persistent) -- Colab's own disk is not
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
except ModuleNotFoundError:
    DRIVE_ROOT = Path("./drive_local")   # off Colab: keep the notebook runnable

PROJECT_DRIVE = DRIVE_ROOT / "NeuroTutorSim"
CACHE_ROOT = PROJECT_DRIVE / "tribe_cache"
HF_CACHE = PROJECT_DRIVE / "hf"
for d in (PROJECT_DRIVE, CACHE_ROOT, HF_CACHE):
    d.mkdir(parents=True, exist_ok=True)
print(f"cache root : {CACHE_ROOT}")
print(f"HF cache   : {HF_CACHE}")


In [ ]:
# 3. HF_HOME -- BEFORE any huggingface import in this process.
#    Set after an import, the ~1 GB checkpoint goes to the ephemeral disk and is
#    re-downloaded every session.
import os
import sys

assert not any(m.startswith(("huggingface_hub", "transformers")) for m in sys.modules), (
    "a HuggingFace module is already imported; Runtime -> Restart session and run this cell first"
)
os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
print("HF_HOME =", os.environ["HF_HOME"])


In [ ]:
# 4. Clone the repo and install the pinned Track A stack into Colab's interpreter
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/MatteoGuardamagna4/neurotutorsim.git"
REPO_DIR = Path("/content/neurotutorsim")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
# --system installs into Colab's own interpreter. `uv sync` would build a venv
# this kernel cannot import from.
#
# `.[tribe,dev]`, NOT `--extra tribe --extra dev`: in pip mode uv rejects --extra
# unless the target is `-r <file>`. With `-e .` it exits 2 with
#   "Requesting extras requires a ... pyproject.toml ... Use <dir>[extra] instead"
subprocess.run(
    ["uv", "pip", "install", "--system", "-e", ".[tribe,dev]"],
    cwd=REPO_DIR,
    check=True,
)

sys.path.insert(0, str(REPO_DIR))
print("installed; Runtime -> Restart session, then continue BELOW this cell")

In [ ]:
# 5. Authenticate. HF_TOKEN comes from Colab Secrets (the key icon), never a literal.
#    `meta-llama/Llama-3.2-3B` (TRIBE's text encoder) is gated per account:
#    a read token does not grant access until Meta approves the request.
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))
print("authenticated")


In [ ]:
# Gate 17 must already have passed for the pinned revision, or the script stops.
!cd /content/neurotutorsim && python -c "\
from src.tribe.config import load_config; from src.tribe import verification; \
c = load_config('config/tribe.yaml'); \
print('gate_17_passed =', verification.gate_17_passed(c))"


In [ ]:
!cd /content/neurotutorsim && python scripts/run_tribe_inference.py \
    --config config/tribe.yaml --cache-root "$CACHE_ROOT"


## Optional: determinism check (§10.1)

Runs one stimulus twice and requires **bitwise-equal** output. Not a tolerance
check — if it fails, stop and report it rather than relaxing the test.


In [ ]:
!cd /content/neurotutorsim && python scripts/run_tribe_inference.py \
    --config config/tribe.yaml --cache-root "$CACHE_ROOT" --check-determinism


## Before closing: verify the manifest

Reports entries recorded but missing on disk, and files on disk missing from
the manifest. It deletes nothing — every entry cost a GPU run against a gated
model.


In [ ]:
!cd /content/neurotutorsim && python scripts/run_tribe_inference.py \
    --config config/tribe.yaml --cache-root "$CACHE_ROOT" --verify-manifest
